# Module 7: EvalOps & AI Telemetry for Continuous Quality

Build an evaluation gate that blocks deployment when model quality regresses.


## 1. Setup

Load EvalOps core components.


In [ ]:
import sys
from pathlib import Path

module_dir = Path.cwd()
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

from evalops_core import make_golden_dataset, SnapAuditPolicyModel, TelemetryCollector, evaluate_model

print("✓ Setup complete")


## 2. Golden Dataset

Create a deterministic set of 100 verified receipt cases.


In [ ]:
dataset = make_golden_dataset(size=100, seed=42)
print("Dataset size:", len(dataset))
dataset[0].to_dict()


## 3. Baseline Evaluation

Baseline model should meet the 99% threshold.


In [ ]:
baseline_model = SnapAuditPolicyModel(variant="baseline")
baseline_telemetry = TelemetryCollector()
baseline_metrics = evaluate_model(dataset, baseline_model, baseline_telemetry)
baseline_metrics


## 4. Regression Simulation

"Friendly" prompt regression that over-approves risky receipts.


In [ ]:
regressed_model = SnapAuditPolicyModel(variant="friendly_regression")
regressed_telemetry = TelemetryCollector()
regressed_metrics = evaluate_model(dataset, regressed_model, regressed_telemetry)
regressed_metrics


## 5. Compare and Diagnose

Inspect failed cases and telemetry traces.


In [ ]:
print("Baseline accuracy:", baseline_metrics["accuracy"])
print("Regressed accuracy:", regressed_metrics["accuracy"])
print("Regression failures:", len(regressed_metrics["failures"]))

regressed_metrics["failures"][:3]


In [ ]:
# Recent telemetry events from regressed run
regressed_telemetry.events[-8:]


## 6. CI Gate Logic

Block deployment if accuracy drops below 99%.


In [ ]:
THRESHOLD = 0.99

def gate(metrics, threshold=THRESHOLD):
    return metrics["accuracy"] >= threshold

print("Baseline gate:", gate(baseline_metrics))
print("Regressed gate:", gate(regressed_metrics))


## 7. Save Artifacts

Persist results and telemetry for observability.


In [ ]:
import json

Path("eval_results_baseline.json").write_text(json.dumps(baseline_metrics, indent=2))
Path("eval_results_regressed.json").write_text(json.dumps(regressed_metrics, indent=2))
Path("telemetry_regressed.json").write_text(json.dumps(regressed_telemetry.events, indent=2))

print("✓ Saved eval and telemetry artifacts")
